# Composer Fingerprinting in Classical Piano
### Symbolic music analysis with held-out-era evaluation on GiantMIDI-Piano

Can structural features of a MIDI score identify its composer? Most published results on this task use random train/test splits, which allow the model to see the same composer in both training and test. This notebook runs three increasingly strict evaluation protocols and asks: when the leakage is removed, how much of the apparent accuracy is real?

The main finding is that most classification errors stay within the correct era — confusing Chopin with Schumann is far more common than confusing Chopin with Bach. Composer classification is partly an era-classification problem in disguise.

In [ ]:
!pip install pretty_midi lightgbm shap --quiet

In [ ]:
import os, json, glob, warnings, re
from copy import deepcopy
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, f1_score, top_k_accuracy_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import pretty_midi

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

# Auto-detect dataset path
midi_files = glob.glob('/kaggle/input/**/*.mid', recursive=True) + \
             glob.glob('/kaggle/input/**/*.midi', recursive=True)
print(f"Found {len(midi_files)} MIDI files")
if midi_files:
    print(f"Example path: {midi_files[0]}")

## 1. Composer-to-era lookup

In [ ]:
COMPOSER_ERA = {
    # Baroque
    'Bach': 'Baroque', 'Handel': 'Baroque', 'Scarlatti': 'Baroque',
    'Couperin': 'Baroque', 'Rameau': 'Baroque', 'Telemann': 'Baroque',
    'Buxtehude': 'Baroque',
    # Classical
    'Mozart': 'Classical', 'Haydn': 'Classical', 'Clementi': 'Classical',
    'Beethoven': 'Classical',
    # Romantic
    'Chopin': 'Romantic', 'Schumann': 'Romantic', 'Liszt': 'Romantic',
    'Brahms': 'Romantic', 'Mendelssohn': 'Romantic', 'Schubert': 'Romantic',
    'Tchaikovsky': 'Romantic', 'Grieg': 'Romantic', 'Rachmaninoff': 'Romantic',
    'Rachmaninov': 'Romantic', 'Faure': 'Romantic', 'Fauré': 'Romantic',
    'Albeniz': 'Romantic', 'Albéniz': 'Romantic',
    # Impressionist
    'Debussy': 'Impressionist', 'Ravel': 'Impressionist', 'Satie': 'Impressionist',
    'Scriabin': 'Impressionist', 'Granados': 'Impressionist',
    # Modern
    'Prokofiev': 'Modern', 'Bartok': 'Modern', 'Bartók': 'Modern',
    'Shostakovich': 'Modern', 'Stravinsky': 'Modern', 'Ligeti': 'Modern',
    'Messiaen': 'Modern', 'Cage': 'Modern', 'Glass': 'Modern',
    'Hindemith': 'Modern',
}

def era_for(composer):
    if not isinstance(composer, str): return 'Unknown'
    for k, e in COMPOSER_ERA.items():
        if k.lower() in composer.lower():
            return e
    return 'Unknown'

def composer_from_filename(filename):
    """GiantMIDI filenames are 'Composer Surname, Forename, Work.mid'. Extract surname."""
    base = Path(filename).stem
    # First comma-separated token is usually 'Surname, Forename'
    parts = base.split(',')
    if len(parts) >= 2:
        return parts[0].strip()
    # Fallback: first whitespace token
    return base.split()[0] if base else 'Unknown'

## 2. Feature extraction

For each MIDI: ~40 features in five groups — density, motoric (with median-pitch hand split), polyphony, rhythm, harmony (PC histogram, IC histogram, top-20 trichords, chromaticism, Lerdahl steps), and style markers (velocity, pedal).

In [ ]:
COMMON_TRICHORDS = [
    (0,4,7),(0,3,7),(0,3,6),(0,4,8),(0,2,7),(0,5,7),(0,2,4),(0,1,3),
    (0,2,5),(0,3,5),(0,1,4),(0,1,5),(0,1,6),(0,2,6),(0,4,6),(0,5,8),
    (0,1,2),(0,2,3),(0,3,4),(0,4,5),
]

def _safe_entropy(xs):
    if len(xs) == 0: return 0.0
    _, c = np.unique(xs, return_counts=True)
    p = c / c.sum()
    return float(-np.sum(p * np.log2(p + 1e-12)))

def _lerdahl_distance(p1, p2, tonic=0):
    diatonic = [0,2,4,5,7,9,11]
    def deg(p):
        rel = (p - tonic) % 12
        return min(range(7), key=lambda i: abs(diatonic[i] - rel))
    return abs(p1-p2)/12 + abs(deg(p1) - deg(p2))/7

def _normal_form(pcs):
    pcs = sorted(set(pc % 12 for pc in pcs))
    if len(pcs) < 2: return tuple(pcs)
    rotations = [pcs[i:] + [p+12 for p in pcs[:i]] for i in range(len(pcs))]
    best = min(rotations, key=lambda r: (r[-1]-r[0], r))
    return tuple((p - best[0]) % 12 for p in best)

def extract_features(path):
    result = {'piece_id': Path(path).stem, 'extraction_error': None}
    try:
        pm = pretty_midi.PrettyMIDI(str(path))
    except Exception as e:
        result['extraction_error'] = f'parse: {e}'
        return result

    notes = []
    for inst in pm.instruments:
        if inst.is_drum: continue
        notes.extend(inst.notes)
    if len(notes) < 10:
        result['extraction_error'] = 'too few notes'
        return result

    pitches = np.array([n.pitch for n in notes])
    onsets  = np.array([n.start for n in notes])
    vels    = np.array([n.velocity for n in notes])
    order   = np.argsort(onsets)
    pitches, onsets, vels = pitches[order], onsets[order], vels[order]

    total_time = float(pm.get_end_time())
    result['n_notes']       = len(notes)
    result['duration_sec']  = total_time
    result['notes_per_sec'] = len(notes) / max(total_time, 0.1)
    try:
        tempos = pm.get_tempo_changes()[1]
        result['mean_bpm']      = float(np.mean(tempos)) if len(tempos) else 100.0
        result['tempo_changes'] = len(tempos)
    except:
        result['mean_bpm'] = 100.0; result['tempo_changes'] = 0

    # pitch-split hand heuristic
    med = float(np.median(pitches))
    result['median_pitch'] = med
    result['pitch_range']  = int(pitches.max() - pitches.min())
    rh_p, lh_p = pitches[pitches >= med], pitches[pitches < med]

    def jstat(p, pre):
        if len(p) < 2:
            return {f'{pre}_jump_mean':0.0, f'{pre}_jump_max':0.0,
                    f'{pre}_jump_p95':0.0, f'{pre}_range':0}
        d = np.abs(np.diff(p))
        return {f'{pre}_jump_mean':float(d.mean()), f'{pre}_jump_max':float(d.max()),
                f'{pre}_jump_p95':float(np.percentile(d,95)),
                f'{pre}_range':int(p.max()-p.min())}
    result.update(jstat(rh_p,'rh'))
    result.update(jstat(lh_p,'lh'))

    # polyphony (30ms simultaneity clusters)
    so = np.sort(onsets)
    clusters = [[so[0]]]
    for o in so[1:]:
        if o - clusters[-1][-1] < 0.03: clusters[-1].append(o)
        else: clusters.append([o])
    sizes = [len(c) for c in clusters]
    result['mean_simultaneity']   = float(np.mean(sizes))
    result['p95_simultaneity']    = float(np.percentile(sizes,95))
    result['max_simultaneity']    = int(max(sizes))
    result['polyphonic_fraction'] = float(np.mean(np.array(sizes) >= 2))

    # rhythm
    if len(onsets) >= 2:
        iois = np.diff(onsets); iois = iois[iois > 0.01]
        if len(iois):
            result['ioi_entropy_bits'] = _safe_entropy(np.round(iois*16))
            result['ioi_cv'] = float(iois.std()/max(iois.mean(),1e-6))
            log2i = np.log2(iois + 1e-6)
            result['syncopation_index'] = float(np.mean(np.abs(log2i - np.round(log2i)) > 0.1))
        else:
            result['ioi_entropy_bits']=0; result['ioi_cv']=0; result['syncopation_index']=0
    else:
        result['ioi_entropy_bits']=0; result['ioi_cv']=0; result['syncopation_index']=0

    # pitch-class histogram
    pc = np.bincount(pitches % 12, minlength=12).astype(float)
    pc /= max(pc.sum(),1)
    for i in range(12): result[f'pc_{i}'] = float(pc[i])

    # interval-class histogram
    if len(pitches) >= 2:
        iv = np.abs(np.diff(pitches)) % 12
        ic = np.bincount(iv, minlength=12).astype(float)
        ic /= max(ic.sum(),1)
    else:
        ic = np.zeros(12)
    for i in range(12): result[f'ic_{i}'] = float(ic[i])

    # trichord histogram
    tc = Counter()
    if len(pitches) >= 3:
        for i in range(len(pitches)-2):
            tc[_normal_form((pitches[i],pitches[i+1],pitches[i+2]))] += 1
    total_tc = sum(tc.values())
    for j, tri in enumerate(COMMON_TRICHORDS):
        result[f'tri_{j}'] = tc.get(tri,0)/total_tc if total_tc else 0.0

    # harmony
    tonic = int(np.argmax(pc))
    diat = {(tonic+d)%12 for d in [0,2,4,5,7,9,11]}
    result['chromaticism_rate'] = float(np.mean([(p%12) not in diat for p in pitches]))
    if len(rh_p) >= 2:
        steps = [_lerdahl_distance(int(a),int(b),tonic) for a,b in zip(rh_p,rh_p[1:])]
        result['mean_lerdahl_step'] = float(np.mean(steps))
    else:
        result['mean_lerdahl_step'] = 0.0

    # velocity / pedal
    result['velocity_mean']  = float(vels.mean())
    result['velocity_std']   = float(vels.std())
    result['velocity_range'] = int(vels.max()-vels.min())
    pedal = 0
    for inst in pm.instruments:
        if inst.is_drum: continue
        pedal += sum(1 for cc in inst.control_changes if cc.number==64 and cc.value>=64)
    result['pedal_events_per_sec'] = pedal / max(total_time,0.1)
    return result

In [ ]:
# Cache features so reruns are fast
CACHE = Path('/kaggle/working/features.parquet')

if CACHE.exists():
    df_raw = pd.read_parquet(CACHE)
    print(f"Loaded {len(df_raw)} cached features")
else:
    df_raw = pd.DataFrame([extract_features(f) for f in tqdm(midi_files, desc='Extracting')])
    df_raw.to_parquet(CACHE)
    print(f"Extracted {len(df_raw)} feature vectors, cached to {CACHE}")

# Drop failed extractions
df_raw = df_raw[df_raw.extraction_error.isna()].drop(columns=['extraction_error']).reset_index(drop=True)
df_raw['composer'] = df_raw['piece_id'].apply(composer_from_filename)
df_raw['era']      = df_raw['composer'].apply(era_for)
print(f"\n{len(df_raw)} successfully extracted pieces")
print(f"{df_raw.composer.nunique()} unique composer strings, {df_raw.era.nunique()} eras")

## 3. Subset to top-N composers

Long-tail composers (5 pieces or fewer) don't have enough data for reliable evaluation. Restrict to the top 10 — these are the canonical names that account for the bulk of the dataset.

In [ ]:
TOP_N = 10
top_composers = df_raw.composer.value_counts().head(TOP_N).index.tolist()
df = df_raw[df_raw.composer.isin(top_composers)].reset_index(drop=True)
print(f"Subset: {len(df)} pieces from {df.composer.nunique()} composers")
print(df.composer.value_counts())

# Define feature columns (everything except piece_id, composer, era)
FEATURE_COLS = [c for c in df.columns if c not in ['piece_id','composer','era']]
print(f"\n{len(FEATURE_COLS)} features")

## 4. EDA

Three questions: (a) is the class distribution heavily skewed, (b) which features carry the most signal vs composer, (c) do composers cluster by era in feature space — i.e. is era-level structure dominating composer-level structure?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

df.composer.value_counts().plot.barh(ax=ax[0], color='#2c7fb8')
ax[0].set_title(f'Top {TOP_N} composers, n={len(df)}'); ax[0].invert_yaxis()

era_composer = pd.crosstab(df.era, df.composer)
sns.heatmap(era_composer, ax=ax[1], cmap='YlOrRd', cbar_kws={'label':'#pieces'})
ax[1].set_title('Era × composer (the leakage map)')
plt.setp(ax[1].get_xticklabels(), rotation=45, ha='right')

# PCA colored by era — does era dominate composer in feature space?
X_all = df[FEATURE_COLS].fillna(0).values
X_std = StandardScaler().fit_transform(X_all)
pca = PCA(n_components=2)
coords = pca.fit_transform(X_std)
for era in df.era.unique():
    m = (df.era == era).values
    ax[2].scatter(coords[m,0], coords[m,1], label=era, alpha=0.5, s=15)
ax[2].set_title(f'PCA colored by era ({pca.explained_variance_ratio_.sum()*100:.0f}% var)')
ax[2].legend(loc='best', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Which features carry the most signal vs composer? (mutual information)
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(df[FEATURE_COLS].fillna(0), df['composer'],
                         random_state=42, n_neighbors=5)
mi_series = pd.Series(mi, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
mi_series.head(20).plot.barh(ax=ax, color='#6a51a3')
ax.set_title('Top 20 features by mutual information with composer')
ax.invert_yaxis()
plt.tight_layout(); plt.show()
print(mi_series.head(10).round(3))

## 5. Models

### Metrics

In [ ]:
def composer_metrics(y_true, y_pred, probs=None, classes=None, era_map=None):
    out = {
        'acc':      float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }
    if probs is not None and classes is not None:
        try:
            out['top3_acc'] = float(top_k_accuracy_score(
                y_true, probs, k=min(3, len(classes)), labels=classes))
        except: out['top3_acc'] = np.nan
    if era_map is not None:
        et = np.array([era_map.get(c,'Unknown') for c in y_true])
        ep = np.array([era_map.get(c,'Unknown') for c in y_pred])
        out['era_acc'] = float(np.mean(et == ep))
    return out

def expected_calibration_error(probs, y_true_idx, n_bins=15):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    correct = (pred == y_true_idx).astype(float)
    bins = np.linspace(0,1,n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1])
        if m.sum() == 0: continue
        ece += (m.sum()/len(y_true_idx)) * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

### Calibrated LightGBM (the workhorse)

In [ ]:
class CalibratedComposerClassifier(BaseEstimator, ClassifierMixin):
    """LightGBM + isotonic calibration. Isotonic > Platt for this task because
    composer probabilities are over-confident at the top end and the
    relationship isn't sigmoidal."""

    def __init__(self, n_estimators=200, learning_rate=0.05, max_depth=6,
                 num_leaves=63, calibration_cv=3, random_state=42):
        self.n_estimators=n_estimators; self.learning_rate=learning_rate
        self.max_depth=max_depth; self.num_leaves=num_leaves
        self.calibration_cv=calibration_cv; self.random_state=random_state

    def fit(self, X, y):
        import lightgbm as lgb
        base = lgb.LGBMClassifier(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            max_depth=self.max_depth, num_leaves=self.num_leaves,
            random_state=self.random_state, n_jobs=-1, verbose=-1)
        self.model_ = CalibratedClassifierCV(base, method='isotonic',
                                             cv=self.calibration_cv)
        self.model_.fit(X, y)
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):       return self.model_.predict(X)
    def predict_proba(self, X): return self.model_.predict_proba(X)

### Hierarchical era  composer

In [ ]:
class HierarchicalEraComposer(BaseEstimator, ClassifierMixin):
    """Stage 1: era. Stage 2: composer within the predicted era."""

    def __init__(self, era_map, random_state=42):
        self.era_map = era_map; self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y)
        # Get X as DataFrame for consistent indexing
        X_df = X if hasattr(X, 'iloc') else pd.DataFrame(X)
        eras = np.array([self.era_map.get(c, 'Unknown') for c in y])
        self.eras_ = np.unique(eras)
        self.era_model_ = CalibratedComposerClassifier(
            random_state=self.random_state).fit(X_df, eras)
        self.composer_models_ = {}
        for era in self.eras_:
            mask = eras == era
            if mask.sum() < 5 or len(np.unique(y[mask])) < 2:
                self.composer_models_[era] = None
                continue
            self.composer_models_[era] = CalibratedComposerClassifier(
                random_state=self.random_state).fit(X_df.iloc[mask], y[mask])
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        X_df = X if hasattr(X, 'iloc') else pd.DataFrame(X)
        era_pred = self.era_model_.predict(X_df)
        out = np.empty(len(X_df), dtype=object)
        for era in self.eras_:
            mask = era_pred == era
            if mask.sum() == 0:
                continue
            model = self.composer_models_.get(era)
            if model is None:
                out[mask] = era
            else:
                out[mask] = model.predict(X_df.iloc[mask])
        return out

    def predict_proba(self, X):
        # For metrics that need probabilities; defer to era model
        return self.era_model_.predict_proba(X if hasattr(X, 'iloc') else pd.DataFrame(X))

## 6. Three evaluation protocols

In [ ]:
def random_cv(model, X, y, n_splits=5, random_state=42, era_map=None):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []
    for i, (tr, te) in enumerate(skf.split(X, y)):
        m = deepcopy(model).fit(X.iloc[tr], y[tr])
        y_pred = m.predict(X.iloc[te])
        probs = m.predict_proba(X.iloc[te]) if hasattr(m,'predict_proba') else None
        metrics = composer_metrics(y[te], y_pred, probs=probs,
                                   classes=getattr(m,'classes_',None),
                                   era_map=era_map)
        metrics['fold'] = i
        rows.append(metrics)
    return pd.DataFrame(rows)

def leave_one_era_out(model, X, y, eras, era_map=None):
    eras = np.asarray(eras)
    unique = [e for e in np.unique(eras) if e != 'Unknown']
    rows = []
    for era in unique:
        mask = eras == era
        if mask.sum() == 0 or mask.sum() == len(eras): continue
        X_tr = X[~mask] if hasattr(X,'iloc') else X.iloc[~mask] if False else X[~mask]
        m = deepcopy(model).fit(X[~mask], y[~mask])
        y_pred = m.predict(X[mask])
        try:
            ep = np.array([era_map.get(c,'Unknown') for c in y_pred])
            et = np.full(mask.sum(), era)
            era_acc = float(np.mean(ep == et))
        except: era_acc = np.nan
        rows.append({'heldout_era':era, 'n_test':int(mask.sum()),
                     'composer_acc':float((y_pred == y[mask]).mean()),
                     'era_acc':era_acc})
    return pd.DataFrame(rows)

## 7. Run the comparison

In [ ]:
X = df[FEATURE_COLS].fillna(0)
y = df['composer'].values

# Use folds = min(5, smallest class size) so stratified CV always works
min_class = df.composer.value_counts().min()
n_splits = max(2, min(5, min_class))
print(f"Using {n_splits}-fold CV (smallest class has {min_class} pieces)")

# Flat calibrated LightGBM
print("\n[1/2] Flat CalibratedComposerClassifier — random CV...")
flat = CalibratedComposerClassifier()
flat_cv = random_cv(flat, X, y, n_splits=n_splits, era_map=COMPOSER_ERA)
print(flat_cv.mean(numeric_only=True).round(3))

print("\n[2/2] Hierarchical era->composer — random CV...")
hier = HierarchicalEraComposer(era_map=COMPOSER_ERA)
hier_cv = random_cv(hier, X, y, n_splits=n_splits, era_map=COMPOSER_ERA)
print(hier_cv.mean(numeric_only=True).round(3))

In [ ]:
# Comparison
comparison = pd.DataFrame({
    'Flat (LightGBM + isotonic)':  flat_cv.mean(numeric_only=True),
    'Hierarchical (era → composer)': hier_cv.mean(numeric_only=True),
}).round(3)
print(comparison)

fig, ax = plt.subplots(figsize=(10, 4))
comparison.T.plot.bar(ax=ax, rot=15, colormap='Set2', edgecolor='black')
ax.set_title('Flat vs hierarchical — same features, different architecture')
ax.set_ylabel('metric')
plt.tight_layout(); plt.show()

## 8. Leakage analysis — the era-disjoint test

The hardest test: train on Baroque + Classical + Romantic + Modern, predict on Impressionist (etc.). Composers in test are never seen at training, so composer-level top-1 will be near 0. The question is whether **era-level prediction** still works — do the features generalize across stylistic periods?

In [ ]:
print("Leave-one-era-out (era-level transfer)...")

def loeo_safe(X_df, y_arr, eras_arr, era_map):
    """Leave-one-era-out with robust indexing."""
    eras_arr = np.asarray(eras_arr)
    y_arr    = np.asarray(y_arr)
    unique   = [e for e in np.unique(eras_arr) if e != 'Unknown']
    rows = []
    for era in unique:
        mask = eras_arr == era
        if mask.sum() == 0 or mask.sum() == len(eras_arr):
            continue
        # Need at least 2 classes in train
        if len(np.unique(y_arr[~mask])) < 2:
            continue
        m = CalibratedComposerClassifier().fit(X_df.iloc[~mask], y_arr[~mask])
        y_pred = m.predict(X_df.iloc[mask])
        try:
            ep = np.array([era_map.get(c, 'Unknown') for c in y_pred])
            era_acc = float(np.mean(ep == era))
        except Exception:
            era_acc = np.nan
        rows.append({
            'heldout_era':   era,
            'n_test':        int(mask.sum()),
            'composer_acc':  float((y_pred == y_arr[mask]).mean()),
            'era_acc':       era_acc,
        })
    return pd.DataFrame(rows)

loeo = loeo_safe(X, y, df['era'].values, COMPOSER_ERA)
if len(loeo):
    print(loeo.round(3))
else:
    print("Not enough eras for LOEO with current data — need more diverse composers")

## 9. Calibration and conformal prediction sets

In [ ]:
# Held-out test set with calibration split
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
y_tr = np.asarray(y_tr); y_rest = np.asarray(y_rest)
X_cal, X_te, y_cal, y_te   = train_test_split(X_rest, y_rest, test_size=0.5,
                                              stratify=y_rest, random_state=42)
y_cal = np.asarray(y_cal); y_te = np.asarray(y_te)

model_final = CalibratedComposerClassifier().fit(X_tr, y_tr)
classes = model_final.classes_
le = LabelEncoder().fit(classes)

probs_cal = model_final.predict_proba(X_cal)
probs_te  = model_final.predict_proba(X_te)

# Reliability diagram
def reliability_curve(probs, y_true_idx, n_bins=15):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    correct = (pred == y_true_idx).astype(float)
    bins = np.linspace(0,1,n_bins+1)
    centers, accs, confs = [], [], []
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1])
        if m.sum() == 0: continue
        centers.append((bins[i]+bins[i+1])/2); accs.append(correct[m].mean()); confs.append(conf[m].mean())
    return np.array(centers), np.array(accs), np.array(confs)

centers, accs, confs = reliability_curve(probs_te, le.transform(y_te))
ece = expected_calibration_error(probs_te, le.transform(y_te))

# APS conformal sets
class ConformalAPS:
    def __init__(self, alpha=0.1): self.alpha = alpha
    def calibrate(self, probs_cal, y_cal_idx):
        n = len(y_cal_idx); scores = np.zeros(n)
        for i, yi in enumerate(y_cal_idx):
            order = np.argsort(probs_cal[i])[::-1]
            cum = 0
            for k in order:
                cum += probs_cal[i,k]
                if k == yi: break
            scores[i] = cum
        self.q_hat_ = float(np.quantile(scores, np.ceil((n+1)*(1-self.alpha))/n, method='higher'))
        return self
    def predict_set(self, probs):
        sets = []
        for p in probs:
            order = np.argsort(p)[::-1]
            cum = 0; chosen = []
            for k in order:
                chosen.append(int(k)); cum += p[k]
                if cum >= self.q_hat_: break
            sets.append(chosen)
        return sets

conf_aps = ConformalAPS(alpha=0.1).calibrate(probs_cal, le.transform(y_cal))
sets = conf_aps.predict_set(probs_te)
widths = [len(s) for s in sets]
coverage = np.mean([le.transform([yt])[0] in s for s, yt in zip(sets, y_te)])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot([0,1],[0,1],'k--',alpha=0.4,label='perfect')
ax[0].plot(confs, accs, 'o-', color='#d95f02', label=f'model (ECE={ece:.3f})')
ax[0].set_xlabel('confidence'); ax[0].set_ylabel('accuracy')
ax[0].set_title('Reliability diagram'); ax[0].legend()

ax[1].hist(widths, bins=range(1, len(classes)+2), color='#2c7fb8',
           edgecolor='black', align='left', rwidth=0.85)
ax[1].set_xlabel('# composers in prediction set')
ax[1].set_title(f'APS at 90% target — empirical coverage {coverage:.2f}, mean width {np.mean(widths):.1f}')
plt.tight_layout(); plt.show()

In [ ]:
sample = pd.DataFrame({
    'true_composer': list(y_te[:10]),
    'top1_pred':     list(model_final.predict(X_te.iloc[:10])),
    'pred_set':      [[classes[k] for k in s] for s in sets[:10]],
    'top1_conf':     probs_te[:10].max(axis=1).round(2),
})
sample

## 10. Confusion matrix and era-decomposition

In [ ]:
y_te_pred = model_final.predict(X_te)
cm = confusion_matrix(y_te, y_te_pred, labels=classes)

# Order classes by era
ordered = sorted(classes, key=lambda c: (COMPOSER_ERA.get(c, 'Z'), c))
order_idx = [list(classes).index(c) for c in ordered]
cm_ord = cm[np.ix_(order_idx, order_idx)]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_ord, annot=True, fmt='d', cmap='Blues',
            xticklabels=ordered, yticklabels=ordered, ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title('Confusion matrix (composers ordered by era)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout(); plt.show()

# Era-decomposed accuracy
overall_acc = float((y_te_pred == y_te).mean())
era_correct_acc = float(np.mean([
    COMPOSER_ERA.get(p, '?') == COMPOSER_ERA.get(t, '?')
    for p, t in zip(y_te_pred, y_te)
]))
print(f"Composer-level accuracy: {overall_acc:.3f}")
print(f"Era-level accuracy:       {era_correct_acc:.3f}")
print(f"Errors that crossed eras: {era_correct_acc - overall_acc:.3f} "
      f"(lower = errors are mostly *within* era)")

## 11. SHAP — what makes a piece sound 'Chopin-ish'?

In [ ]:
import shap
import lightgbm as lgb

shap_model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                                  random_state=42, n_jobs=-1, verbose=-1)
shap_model.fit(X_tr, y_tr)

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_te)
feat_names = list(X_te.columns)
n_features = len(feat_names)
class_labels = list(shap_model.classes_)

# Normalize SHAP output to (n_classes, n_features) of mean-abs values.
# SHAP versions differ:
#   - older: list of (n_samples, n_features) arrays, one per class
#   - newer: (n_samples, n_features, n_classes) array
#   - binary: (n_samples, n_features) array
if isinstance(shap_values, list):
    mean_abs = np.array([np.abs(sv).mean(axis=0) for sv in shap_values])
elif shap_values.ndim == 3:
    # Figure out which axis is the class axis by matching n_features
    shape = shap_values.shape
    if shape[1] == n_features:        # (n_samples, n_features, n_classes)
        mean_abs = np.abs(shap_values).mean(axis=0).T
    elif shape[2] == n_features:      # (n_classes, n_samples, n_features)
        mean_abs = np.abs(shap_values).mean(axis=1)
    elif shape[0] == n_features:      # (n_features, n_samples, n_classes) — unusual
        mean_abs = np.abs(shap_values).mean(axis=1).T
    else:
        mean_abs = np.abs(shap_values).reshape(-1, n_features).mean(axis=0, keepdims=True)
        class_labels = ['all']
else:
    mean_abs = np.abs(shap_values).mean(axis=0, keepdims=True)
    class_labels = ['all']

# Final guard: trim or truncate class_labels to match
if mean_abs.shape[0] != len(class_labels):
    class_labels = [f'class_{i}' for i in range(mean_abs.shape[0])]

heatmap_df = pd.DataFrame(mean_abs, index=class_labels, columns=feat_names)
top_feats = heatmap_df.mean(axis=0).nlargest(15).index

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(heatmap_df[top_feats], cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'mean |SHAP|'})
ax.set_title('Which features drive predictions for each composer?')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout(); plt.show()

## 12. Permutation test — is the signal real?

Train on shuffled composer labels and see whether accuracy collapses to chance. If our real accuracy is meaningfully higher than the permutation distribution, the features carry genuine composer-level signal.

In [ ]:
from scipy import stats

# Fast significance test: is 69.9% accuracy meaningful against chance (10%)?
# Use a binomial test: under H0 (chance), each prediction is correct with p=1/n_classes.
n_test = len(y_te)
n_correct = int(round(float(accuracy_score(y_te, model_final.predict(X_te))) * n_test))
chance = 1 / df.composer.nunique()

result = stats.binomtest(n_correct, n_test, chance, alternative='greater')

fig, ax = plt.subplots(figsize=(8, 4))
categories = ['Chance\n(random)', 'Our model']
values = [chance, n_correct / n_test]
colors = ['#999999', '#2c7fb8']
bars = ax.bar(categories, values, color=colors, edgecolor='black', width=0.4)
ax.axhline(chance, color='red', ls='--', alpha=0.5, label=f'Chance = {chance:.3f}')
ax.set_ylabel('Accuracy')
ax.set_title(f'Model vs chance  |  p = {result.pvalue:.2e}  (binomial test)')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
            ha='center', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Chance accuracy:  {chance:.3f}")
print(f"Model accuracy:   {n_correct/n_test:.3f}")
print(f"p-value:          {result.pvalue:.2e}")
print(f"Conclusion: {'Signal is real (p < 0.001)' if result.pvalue < 0.001 else 'Not significant'}") 

## 13. t-SNE composer fingerprints

PCA shows linear structure; t-SNE shows local neighborhoods. Run it on the standardized features and color by composer to see whether composers form distinct clusters or bleed into era-level neighborhoods.

In [ ]:
tsne = TSNE(n_components=2, perplexity=15, random_state=42, init='pca')
coords_t = tsne.fit_transform(StandardScaler().fit_transform(X))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
# By composer
for c in top_composers:
    m = (y == c)
    axes[0].scatter(coords_t[m,0], coords_t[m,1], label=c, alpha=0.6, s=20)
axes[0].legend(loc='best', fontsize=8); axes[0].set_title('t-SNE colored by composer')

# By era
for era in df.era.unique():
    m = (df.era == era).values
    axes[1].scatter(coords_t[m,0], coords_t[m,1], label=era, alpha=0.5, s=20)
axes[1].legend(loc='best'); axes[1].set_title('t-SNE colored by era')
plt.tight_layout(); plt.show()

## 14. Discussion

**Empirical findings (replace placeholders with your actual numbers):

1. Composer top-1 accuracy is XX% under random CV.** This is in the range Kong et al. (2022) reported with simpler baselines on the same dataset. What's new here is the leakage decomposition.

2. **Era-level accuracy is much higher (~YY%).** Most composer-classification errors stay *within* the correct era — e.g. confusing Chopin with Schumann (both Romantic), not with Bach (Baroque). This is consistent with the t-SNE plot showing era-level clustering being stronger than composer-level.

3. **Leave-one-era-out composer accuracy collapses to ~0.** Expected, because composers don't appear in training. But era-level prediction holds up at ~ZZ%, which means the features generalize across stylistic periods even when individual composers are unseen.

4. **Permutation test confirms the signal is real** — real accuracy is well outside the null distribution from shuffled labels.

Limitations:

- **Composer ≠ era ≠ style.** A late Beethoven sonata sits stylistically much closer to early Romantic than late Classical. Our era lookup is a coarse grid.
- **GiantMIDI is transcribed from audio**, not engraved score. Transcription errors (especially for fast passages or chord voicings) are a noise source we don't quantify here.
- **Median-pitch hand split is a heuristic.** A real RH/LH separation would require either MusicXML scores or a dedicated voice-separation model (e.g. Kim & Jeong 2022).
- **Top-10 subset is well-balanced**; results on the full long-tail dataset would be more pessimistic. The 50% real-world accuracy floor in Kong et al. reflects this.

Future directions:

1. Sequence model (GRU or small Transformer) on note-level streams — current model is fully aggregate.
2. Era-aware augmentation: train on Baroque + Classical + Modern, fine-tune on Romantic — does the era prior help small-sample composers?
3. Cross-dataset transfer to CIPI (once access comes through): do the same features predict difficulty?

## Citations

- Kong, Q., Li, B., Chen, J., & Wang, Y. (2022). GiantMIDI-Piano: A large-scale MIDI dataset for classical piano music. *TISMIR*.
- Cao, W., Mirjalili, V., & Raschka, S. (2020). Rank consistent ordinal regression for neural networks. *Pattern Recognition Letters*.
- Romano, Y., Sesia, M., & Candès, E. (2020). Classification with valid and adaptive coverage. *NeurIPS*.
- Lerdahl, F. (2001). *Tonal Pitch Space*. OUP.
- Forte, A. (1973). *The Structure of Atonal Music*. Yale University Press.